In [ ]:
56

In [ ]:
import os
import pymupdf4llm

 
image_folder = "report-images"
os.makedirs(image_folder, exist_ok=True)

 
 
data = pymupdf4llm.to_markdown(
    doc="Sample Report.pdf",
    page_chunks=True,        
    write_images=True,       
    image_path=image_folder
)
 


In [ ]:
data[0]

In [ ]:
data[0]['text']

In [ ]:
text_data = [ind['text'] for ind in data]

In [ ]:
text_data[0]

In [ ]:
text_context = "<----------------next------------>".join(text_data)

In [ ]:
print(text_context)

In [ ]:
def extract_text_images(doc_path:str,image_folder:str):
 
    os.makedirs(image_folder, exist_ok=True)   
    data = pymupdf4llm.to_markdown(
        doc=doc_path,
        page_chunks=True,        
        write_images=True,       
        image_path=image_folder
    )
    
    text_data = [ind['text'] for ind in data]
    text_context = "<----------------next------------>".join(text_data)

    return text_context

In [ ]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()
import base64
model1 = ChatGroq(model="openai/gpt-oss-120b")
model2 = ChatOpenAI(model="gpt-4o-mini")

In [ ]:

report_prompt = f"""You are a Text Diagnosis Agent for structural inspection reports.

INPUT:
You will receive raw extracted text from an inspection PDF along with image paths already mapped to sections.

OBJECTIVE:
Convert the raw text into a clean, structured, area-wise diagnostic JSON.
Improve clarity and structure WITHOUT changing meaning.
DO NOT lose any information.
DO NOT hallucinate.

STRICT RULES:
- Only use information present in the input
- If something is missing → write "Not Available"
- If information is unclear → write "Unclear"
- Do NOT assume or infer beyond given text
- Preserve all image paths exactly as provided
- Do NOT remove or rename image paths
- Do NOT merge unrelated areas
- Avoid duplication
- Keep language simple and client-friendly

TASKS:
1. Identify all areas (Hall, Bedroom, Kitchen, etc.)
2. Extract observations/issues per area
3. Clean and standardize descriptions
4. Attach corresponding image paths to correct areas
5. Identify any mentioned causes (if explicitly present)
6. Identify severity hints ONLY if clearly mentioned
7. Capture missing or unclear information

OUTPUT FORMAT (STRICT JSON ONLY):

{{
  "source": "inspection",
  "areas": [
    {{
      "area": "",
      "observations": [
        {{
          "issue": "",
          "description": "",
          "severity_hint": "Low | Medium | High | Not Available",
          "images": []
        }}
      ]
    }}
  ],
  "general_notes": [],
  "missing_information": []
}}

IMPORTANT:
- Do not add extra fields
- Do not output anything except JSON
- Ensure no data loss from input

Extracted text:
{text_context}

"""
result = model1.invoke(report_prompt)

In [ ]:
print(result.content)

In [ ]:
def report_text_diagnosis():
    report_prompt = f"""You are a Text Diagnosis Agent for structural inspection reports.

    INPUT:
    You will receive raw extracted text from an inspection PDF along with image paths already mapped to sections.

    OBJECTIVE:
    Convert the raw text into a clean, structured, area-wise diagnostic JSON.
    Improve clarity and structure WITHOUT changing meaning.
    DO NOT lose any information.
    DO NOT hallucinate.

    STRICT RULES:
    - Only use information present in the input
    - If something is missing → write "Not Available"
    - If information is unclear → write "Unclear"
    - Do NOT assume or infer beyond given text
    - Preserve all image paths exactly as provided
    - Do NOT remove or rename image paths
    - Do NOT merge unrelated areas
    - Avoid duplication
    - Keep language simple and client-friendly

    TASKS:
    1. Identify all areas (Hall, Bedroom, Kitchen, etc.)
    2. Extract observations/issues per area
    3. Clean and standardize descriptions
    4. Attach corresponding image paths to correct areas
    5. Identify any mentioned causes (if explicitly present)
    6. Identify severity hints ONLY if clearly mentioned
    7. Capture missing or unclear information

    OUTPUT FORMAT (STRICT JSON ONLY):

    {{
    "source": "inspection",
    "areas": [
        {{
        "area": "",
        "observations": [
            {{
            "issue": "",
            "description": "",
            "severity_hint": "Low | Medium | High | Not Available",
            "images": []
            }}
        ]
        }}
    ],
    "general_notes": [],
    "missing_information": []
    }}

    IMPORTANT:
    - Do not add extra fields
    - Do not output anything except JSON
    - Ensure no data loss from input

    Extracted text:
    {text_context}

    """
    result = model1.invoke(report_prompt)

In [ ]:
def thermal_text_diagnosis():
    prompt = f""" 
    You are a Thermal Analysis Diagnosis Agent.

    INPUT:
    You will receive raw extracted text from a thermal report PDF along with image paths.

    OBJECTIVE:
    Convert thermal readings and observations into structured insights.
    Focus on temperature interpretation and possible issue indicators.
    DO NOT hallucinate.

    STRICT RULES:
    - Only use values explicitly present (hotspot, coldspot, emissivity, etc.)
    - Do NOT invent causes unless logically derivable (e.g., coldspot → possible moisture)
    - If unsure → mark "Low Confidence"
    - If area is not mentioned → "Not Clearly Identified"
    - Preserve all image paths exactly
    - Do NOT drop any readings

    TASKS:
    1. Extract thermal readings (hotspot, coldspot, etc.)
    2. Interpret temperature differences
    3. Identify possible issue indicators (moisture, leakage, heat loss)
    4. Assign confidence level (High/Medium/Low)
    5. Attach image paths

    OUTPUT FORMAT (STRICT JSON ONLY):

    {{
    "source": "thermal",
    "observations": [
        {{
        "image": "",
        "hotspot": "",
        "coldspot": "",
        "interpretation": "",
        "possible_issue": "",
        "confidence": "High | Medium | Low",
        "area": "Not Clearly Identified"
        }}
    ],
    "general_notes": [],
    "missing_information": []
    }}

    IMPORTANT:
    - Do not output anything except JSON
    - Keep interpretations grounded in data
    - Do not exaggerate conclusions

    Extratced text:
    {text_context}
    """
    result = model1.invoke(prompt)

In [ ]:
def encode_pdf_to_base64(file_path):
    with open(file_path, "rb") as pdf_file:
        return base64.b64encode(pdf_file.read()).decode('utf-8')
    
pdf_b64 = encode_pdf_to_base64("Sample Report.pdf")    

In [ ]:
prompt ="""You are a Vision Diagnosis Agent for structural inspection images.

    INPUT:
    You will receive:
    - One pdf consist of many images 
    - The image file path (must be preserved)
    - Optional context text (may or may not be accurate)

    OBJECTIVE:
    for all images in pdf "Analyze the image and extract ONLY visible, verifiable structural issues.
    Do NOT hallucinate or assume anything not clearly visible".

    STRICT RULES:
    - Only describe what is visually observable
    - Do NOT infer hidden causes unless clearly supported
    - If unsure → mark "Unclear"
    - If nothing significant → "No Visible Issue"
    - Preserve the exact image path
    - Do NOT rename or modify image path
    - Do NOT use technical jargon unnecessarily
    - Keep descriptions simple and client-friendly

    TASKS:
    1. Identify visible issues (dampness, cracks, leakage marks, tile gaps, etc.)
    2. Describe the issue clearly
    3. Estimate severity visually (Low / Medium / High)
    4. Classify issue type
    5. Assign confidence level

    OUTPUT FORMAT (STRICT JSON ONLY) for each images:

    {
    "source": "inspection_image",
    "image": "",
    "observations": [
        {
        "issue_type": "Dampness | Crack | Leakage | Tile Gap | Stain | No Visible Issue | Unclear",
        "description": "",
        "severity": "Low | Medium | High | Not Available",
        "confidence": "High | Medium | Low"
        }
    ]
    }

    IMPORTANT:
    - Do not output anything except JSON
    - Do not assume area unless explicitly visible or provided
    - Do not fabricate details"""
    
message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": prompt
            },
            {
                "type": "file",
                "file": {
                    "filename": "document.pdf",
                    "file_data": f"data:application/pdf;base64,{pdf_b64}"
                }
            }
        ]
    )

result1 = model2.invoke([message])


In [ ]:
print(result1.content)

In [ ]:
def impact_image_diagnosis():
        
    prompt ="""You are a Vision Diagnosis Agent for structural inspection images.

        INPUT:
        You will receive:
        - One pdf consist of many images 
        - The image file path (must be preserved)
        - Optional context text (may or may not be accurate)

        OBJECTIVE:
        for all images in pdf "Analyze the image and extract ONLY visible, verifiable structural issues.
        Do NOT hallucinate or assume anything not clearly visible".

        STRICT RULES:
        - Only describe what is visually observable
        - Do NOT infer hidden causes unless clearly supported
        - If unsure → mark "Unclear"
        - If nothing significant → "No Visible Issue"
        - Preserve the exact image path
        - Do NOT rename or modify image path
        - Do NOT use technical jargon unnecessarily
        - Keep descriptions simple and client-friendly

        TASKS:
        1. Identify visible issues (dampness, cracks, leakage marks, tile gaps, etc.)
        2. Describe the issue clearly
        3. Estimate severity visually (Low / Medium / High)
        4. Classify issue type
        5. Assign confidence level

        OUTPUT FORMAT (STRICT JSON ONLY) for each images:

        {
        "source": "inspection_image",
        "image": "",
        "observations": [
            {
            "issue_type": "Dampness | Crack | Leakage | Tile Gap | Stain | No Visible Issue | Unclear",
            "description": "",
            "severity": "Low | Medium | High | Not Available",
            "confidence": "High | Medium | Low"
            }
        ]
        }

        IMPORTANT:
        - Do not output anything except JSON
        - Do not assume area unless explicitly visible or provided
        - Do not fabricate details"""
        
    message = HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": prompt
                },
                {
                    "type": "file",
                    "file": {
                        "filename": "document.pdf",
                        "file_data": f"data:application/pdf;base64,{pdf_b64}"
                    }
                }
            ]
        )

    result1 = model2.invoke([message])


In [ ]:
def thermal_image_diagnosis():

    prompt ="""You are a Thermal Vision Analysis Agent.

        INPUT:
        You will receive:
        - One pdf consist of many images 
        - The image file path (must be preserved)
        - Optional extracted values (hotspot, coldspot)

        OBJECTIVE:
        For all images "Analyze the thermal image and identify temperature patterns and possible issues".

        STRICT RULES:
        - Use visible thermal patterns (color differences, gradients)
        - If hotspot/coldspot values are visible → extract them
        - Do NOT invent numeric values
        - Interpret conservatively
        - If unsure → mark "Low Confidence"
        - If no clear anomaly → "No Significant Thermal Anomaly"
        - Preserve image path exactly

        TASKS:
        1. Identify temperature variation (hot vs cold regions)
        2. Extract hotspot/coldspot if visible
        3. Interpret pattern (uniform / localized variation)
        4. Identify possible issue:
        - Moisture / Leakage (cold regions)
        - Heat loss / insulation issue (hot regions)
        - No anomaly
        5. Assign confidence level

        OUTPUT FORMAT (STRICT JSON ONLY) for each image:

        {
        "source": "thermal_image",
        "image": "",
        "hotspot": "Not Available",
        "coldspot": "Not Available",
        "pattern": "Uniform | Localized Variation | Unclear",
        "interpretation": "",
        "possible_issue": "Moisture | Leakage | Heat Loss | No Significant Issue | Unclear",
        "confidence": "High | Medium | Low",
        "area": "Not Clearly Identified"
        }

        IMPORTANT:
        - Do not output anything except JSON
        - Do not exaggerate conclusions
        - Do not assume area if not given"""

    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": prompt
            },
            {
                "type": "file",
                "file": {
                    "filename": "document.pdf",
                    "file_data": f"data:application/pdf;base64,{pdf_b64}"
                }
            }
        ]
    )


    result1 = model2.invoke([message])


In [ ]:
prompt3 = f"""You are a Merge Agent for an Inspection Report.

    INPUT:
    - text_diagnosis_json  (from Text Diagnosis Agent)
    - image_diagnosis_json_list (array of Vision Agent outputs for inspection images)

    OBJECTIVE:
    Create a unified, area-wise inspection report by merging text and image diagnoses.

    STRICT RULES:
    - Do NOT hallucinate. Use only provided inputs.
    - Do NOT lose any information.
    - Preserve all image paths exactly.
    - If duplicate issues exist → merge them into one entry.
    - If conflicts exist → record them explicitly in "conflicts".
    - If information is missing → write "Not Available".
    - If uncertain → write "Unclear".
    - Do NOT invent areas; use areas from text_diagnosis_json. If an image has no area, attach it to "Unmapped".

    TASKS:
    1) Normalize area names (e.g., "MB Bedroom" → "Master Bedroom") without changing meaning.
    2) For each area:
    - Merge text observations with image observations.
    - Deduplicate similar issues (same type + location).
    - Attach all relevant image paths to the merged issue.
    3) Map image observations:
    - If an image includes area info → map to that area.
    - Else → place under area "Unmapped".
    4) Capture severity:
    - Prefer explicit severity from text; else use image severity; else "Not Available".
    5) Capture conflicts:
    - Example: text says dampness, image shows no visible issue.
    6) Collect general notes and missing information.

    OUTPUT FORMAT (STRICT JSON ONLY):

    {{
    "source": "inspection_merged",
    "areas": [
        {{
        "area": "",
        "observations": [
            {{
            "issue_type": "Dampness | Crack | Leakage | Tile Gap | Stain | Other",
            "description": "",
            "severity": "Low | Medium | High | Not Available",
            "evidence": {{
                "text": [],
                "images": []
            }},
            "confidence": "High | Medium | Low"
            }}
        ]
        }}
    ],
    "unmapped_images": [
        {{
        "image": "",
        "observations": []
        }}
    ],
    "conflicts": [
        {{
        "area": "",
        "detail": ""
        }}
    ],
    "general_notes": [],
    "missing_information": []
    }}

    IMPORTANT:
    - Output JSON only.
    - Ensure every image path from input appears either under an area or in "unmapped_images".
    - Keep language simple and client-friendly.
    
    Input JSONs:
    text_diagnosis_json:
    {result.content}

    image_diagnosis_json
    {result1.content}
    
    """

result2 = model2.invoke(prompt3)

In [ ]:
print(result2.content)

In [ ]:
def incpection_merge():

    prompt3 = f"""You are a Merge Agent for an Inspection Report.

        INPUT:
        - text_diagnosis_json  (from Text Diagnosis Agent)
        - image_diagnosis_json_list (array of Vision Agent outputs for inspection images)

        OBJECTIVE:
        Create a unified, area-wise inspection report by merging text and image diagnoses.

        STRICT RULES:
        - Do NOT hallucinate. Use only provided inputs.
        - Do NOT lose any information.
        - Preserve all image paths exactly.
        - If duplicate issues exist → merge them into one entry.
        - If conflicts exist → record them explicitly in "conflicts".
        - If information is missing → write "Not Available".
        - If uncertain → write "Unclear".
        - Do NOT invent areas; use areas from text_diagnosis_json. If an image has no area, attach it to "Unmapped".

        TASKS:
        1) Normalize area names (e.g., "MB Bedroom" → "Master Bedroom") without changing meaning.
        2) For each area:
        - Merge text observations with image observations.
        - Deduplicate similar issues (same type + location).
        - Attach all relevant image paths to the merged issue.
        3) Map image observations:
        - If an image includes area info → map to that area.
        - Else → place under area "Unmapped".
        4) Capture severity:
        - Prefer explicit severity from text; else use image severity; else "Not Available".
        5) Capture conflicts:
        - Example: text says dampness, image shows no visible issue.
        6) Collect general notes and missing information.

        OUTPUT FORMAT (STRICT JSON ONLY):

        {{
        "source": "inspection_merged",
        "areas": [
            {{
            "area": "",
            "observations": [
                {{
                "issue_type": "Dampness | Crack | Leakage | Tile Gap | Stain | Other",
                "description": "",
                "severity": "Low | Medium | High | Not Available",
                "evidence": {{
                    "text": [],
                    "images": []
                }},
                "confidence": "High | Medium | Low"
                }}
            ]
            }}
        ],
        "unmapped_images": [
            {{
            "image": "",
            "observations": []
            }}
        ],
        "conflicts": [
            {{
            "area": "",
            "detail": ""
            }}
        ],
        "general_notes": [],
        "missing_information": []
        }}

        IMPORTANT:
        - Output JSON only.
        - Ensure every image path from input appears either under an area or in "unmapped_images".
        - Keep language simple and client-friendly.
        
        Input JSONs:
        text_diagnosis_json:
        {result.content}

        image_diagnosis_json
        {result1.content}
        
        """

    result2 = model2.invoke(prompt3)

In [ ]:
def thermal_merge():

    prompt3 = f"""You are a Merge Agent for a Thermal Report.

    INPUT:
    - text_diagnosis_json  (from Thermal Text Diagnosis Agent)
    - image_diagnosis_json_list (array of Vision Agent outputs for thermal images)

    OBJECTIVE:
    Create a unified thermal analysis by combining readings and visual patterns.

    STRICT RULES:
    - Do NOT hallucinate. Use only provided inputs.
    - Do NOT invent numeric values.
    - Preserve all image paths exactly.
    - If values conflict → record in "conflicts".
    - If area is not specified → "Not Clearly Identified".
    - If information is missing → "Not Available".
    - Interpret conservatively.

    TASKS:
    1) For each image:
    - Combine text readings (hotspot/coldspot) with visual analysis (pattern, issue).
    2) Prefer explicit numeric values from text; if absent, use only what is visible.
    3) Derive interpretation:
    - Cold region → possible moisture/leakage
    - Hot region → possible heat loss/insulation issue
    - Uniform → likely no significant anomaly
    4) Assign confidence:
    - High: consistent text + image
    - Medium: partial match
    - Low: unclear/weak signal
    5) Keep each image as a separate observation entry.
    6) Capture conflicts and missing info.

    OUTPUT FORMAT (STRICT JSON ONLY):

    {{
    "source": "thermal_merged",
    "observations": [
        {{
        "image": "",
        "hotspot": "Not Available",
        "coldspot": "Not Available",
        "pattern": "Uniform | Localized Variation | Unclear",
        "interpretation": "",
        "possible_issue": "Moisture | Leakage | Heat Loss | No Significant Issue | Unclear",
        "confidence": "High | Medium | Low",
        "area": "Not Clearly Identified"
        }}
    ],
    "conflicts": [
        {{
        "image": "",
        "detail": ""
        }}
    ],
    "general_notes": [],
    "missing_information": []
    }}

    IMPORTANT:
    - Output JSON only.
    - Ensure every image from input is present in "observations".
    - Do not exaggerate conclusions.
    
    Input JSONs:
    -text_diagnosis_json:
    {result.content}

    -image_diagnosis_json
    {result1.content}
    

    """

    result2 = model2.invoke(prompt3)

In [ ]:
prompt4 = f"""
You are a DDR (Detailed Diagnostic Report) Generation Agent.

INPUT:
You will receive:
1. inspection_merged_json  (merged output of inspection text + images)
2. thermal_merged_json    (merged output of thermal text + images)

OBJECTIVE:
Generate a final, structured, client-ready DDR report by combining both inputs.

STRICT RULES:
- Do NOT hallucinate
- Do NOT invent facts or causes
- Use ONLY provided data
- If information is missing → write "Not Available"
- If data conflicts → explicitly mention in "conflicts"
- Avoid duplicate points
- Keep language simple and client-friendly
- Preserve all image paths exactly
- Ensure images support the corresponding observations
- Do NOT drop any critical information

---

TASKS:

1. PROPERTY ISSUE SUMMARY
- Provide a concise overview of major issues across the property
- Base only on repeated or significant observations

---

2. AREA-WISE OBSERVATIONS
For each area (Hall, Bedroom, Kitchen, etc.):
- Merge inspection + thermal findings
- Avoid duplication
- Attach relevant images
- If thermal data not mapped → include only inspection
- If area unclear → "Not Clearly Identified"

Each observation must include:
- issue
- description
- supporting evidence (text + thermal insight)
- image paths

---

3. PROBABLE ROOT CAUSE
- Only if logically supported by BOTH sources or clearly stated
- If not enough data → "Not Available"

---

4. SEVERITY ASSESSMENT (WITH REASONING)
- Assign: Low / Medium / High
- Based on:
  - frequency of issue
  - structural impact (crack vs stain)
  - thermal anomaly strength
- Provide reasoning

---

5. RECOMMENDED ACTIONS
- Suggest practical fixes based ONLY on identified issues
- Keep simple (repair leakage, seal joints, etc.)
- Do NOT suggest advanced engineering unless clearly needed

---

6. ADDITIONAL NOTES
- General observations
- Patterns (e.g., multiple dampness areas)

---

7. MISSING OR UNCLEAR INFORMATION
- Explicitly list:
  - missing values
  - unclear mappings
  - unavailable images

---

8. CONFLICTS
- Clearly mention contradictions between inspection and thermal data

---

OUTPUT FORMAT (STRICT JSON ONLY):

{{
  "property_issue_summary": "",
  "areas": [
    {{
      "area": "",
      "observations": [
        {{
          "issue": "",
          "description": "",
          "thermal_insight": "Not Available",
          "root_cause": "Not Available",
          "severity": {{
            "level": "Low | Medium | High",
            "reason": ""
          }},
          "recommended_action": "",
          "images": []
        }}
      ]
    }}
  ],
  "additional_notes": [],
  "missing_information": [],
  "conflicts": []
}}

---

IMPORTANT:
- Output ONLY JSON (no explanation)
- Ensure every image path from inputs appears in relevant sections
- Do NOT duplicate the same issue multiple times
- Maintain logical consistency
- Keep wording clear and client-friendly

Input JSONs:

-inspection_merged_json:
{result2.content}

-thermal_merged_json
{result2.content}


"""


result4 = model2.invoke(prompt4)

In [ ]:
print(result4.content)

In [ ]:
def ddr_agent():

    prompt4 = f"""
    You are a DDR (Detailed Diagnostic Report) Generation Agent.

    INPUT:
    You will receive:
    1. inspection_merged_json  (merged output of inspection text + images)
    2. thermal_merged_json    (merged output of thermal text + images)

    OBJECTIVE:
    Generate a final, structured, client-ready DDR report by combining both inputs.

    STRICT RULES:
    - Do NOT hallucinate
    - Do NOT invent facts or causes
    - Use ONLY provided data
    - If information is missing → write "Not Available"
    - If data conflicts → explicitly mention in "conflicts"
    - Avoid duplicate points
    - Keep language simple and client-friendly
    - Preserve all image paths exactly
    - Ensure images support the corresponding observations
    - Do NOT drop any critical information

    ---

    TASKS:

    1. PROPERTY ISSUE SUMMARY
    - Provide a concise overview of major issues across the property
    - Base only on repeated or significant observations

    ---

    2. AREA-WISE OBSERVATIONS
    For each area (Hall, Bedroom, Kitchen, etc.):
    - Merge inspection + thermal findings
    - Avoid duplication
    - Attach relevant images
    - If thermal data not mapped → include only inspection
    - If area unclear → "Not Clearly Identified"

    Each observation must include:
    - issue
    - description
    - supporting evidence (text + thermal insight)
    - image paths

    ---

    3. PROBABLE ROOT CAUSE
    - Only if logically supported by BOTH sources or clearly stated
    - If not enough data → "Not Available"

    ---

    4. SEVERITY ASSESSMENT (WITH REASONING)
    - Assign: Low / Medium / High
    - Based on:
    - frequency of issue
    - structural impact (crack vs stain)
    - thermal anomaly strength
    - Provide reasoning

    ---

    5. RECOMMENDED ACTIONS
    - Suggest practical fixes based ONLY on identified issues
    - Keep simple (repair leakage, seal joints, etc.)
    - Do NOT suggest advanced engineering unless clearly needed

    ---

    6. ADDITIONAL NOTES
    - General observations
    - Patterns (e.g., multiple dampness areas)

    ---

    7. MISSING OR UNCLEAR INFORMATION
    - Explicitly list:
    - missing values
    - unclear mappings
    - unavailable images

    ---

    8. CONFLICTS
    - Clearly mention contradictions between inspection and thermal data

    ---

    OUTPUT FORMAT (STRICT JSON ONLY):

    {{
    "property_issue_summary": "",
    "areas": [
        {{
        "area": "",
        "observations": [
            {{
            "issue": "",
            "description": "",
            "thermal_insight": "Not Available",
            "root_cause": "Not Available",
            "severity": {{
                "level": "Low | Medium | High",
                "reason": ""
            }},
            "recommended_action": "",
            "images": []
            }}
        ]
        }}
    ],
    "additional_notes": [],
    "missing_information": [],
    "conflicts": []
    }}

    ---

    IMPORTANT:
    - Output ONLY JSON (no explanation)
    - Ensure every image path from inputs appears in relevant sections
    - Do NOT duplicate the same issue multiple times
    - Maintain logical consistency
    - Keep wording clear and client-friendly

    Input JSONs:

    -inspection_merged_json:
    {result2.content}

    -thermal_merged_json
    {result2.content}


    """


    result4 = model2.invoke(prompt4)

In [ ]:
import json
from langchain_core.utils.json import parse_json_markdown

parsed_json = parse_json_markdown(result4.content)

with open("ddr.json", "w") as file:
    json.dump(parsed_json, file, indent=4)

In [ ]:
from jinja2 import Template
import json

with open("ddr.json") as f:
    data = json.load(f)

with open("template.html") as f:
    template = Template(f.read())

html = template.render(
    summary=data.get("property_issue_summary"),
    areas=data.get("areas"),
    additional_notes=data.get("additional_notes"),
    missing_information=data.get("missing_information"),
    conflicts=data.get("conflicts")
)

with open("report.html", "w") as f:
    f.write(html)

In [ ]:
import webbrowser
webbrowser.open("report.html")

In [ ]:
from src.extractor import extract_text_images
from src.text_diagnosis import report_text_diagnosis, thermal_text_diagnosis
from src.image_dignosis import impact_image_diagnosis, thermal_image_diagnosis
from src.merger import incpection_merge, thermal_merger
from src.ddr import ddr_agent
from src.state import DDRState
from src.last_node import save_json


from langgraph.graph import StateGraph, START, END

def impact_extraction(state:DDRState) -> DDRState:
    file_path = state['inspection_path']
    text_context = extract_text_images(doc_path=file_path,image_folder="inspect")
    return {
        'raw_inspection_text' : text_context
    }



def thermal_extraction(state:DDRState) -> DDRState:
    file_path = state['thermal_path']
    text_context = extract_text_images(doc_path=file_path,image_folder="thermal")
    return {
        'raw_thermal_text' : text_context
    }


def impact_text(state:DDRState) -> DDRState:
    text_context = state['raw_inspection_text']
    result = report_text_diagnosis(text_context=text_context)
    return{
        'inpection_text_dignosis':result
    }



def impact_image(state:DDRState) -> DDRState:
    file_path = state['inspection_path']
    result = impact_image_diagnosis(file_path=file_path)
    return {
        'inspection_image_dignosis':result
    }


def thermal_text(state:DDRState) -> DDRState:
    text_context = state['raw_thermal_text']
    result = thermal_text_diagnosis(text_context)
    return{
        'thermal_text_dignosis':result
    }


def thermal_image(state:DDRState) -> DDRState:
    file_path = state['thermal_path']
    result = thermal_image_diagnosis(file_path=file_path)
    return {
        'thermal_image_diagnosis':result
    }

def impact_merge(state:DDRState) -> DDRState:
    text = state['inpection_text_dignosis']
    image = state['inspection_image_dignosis']
    result = incpection_merge(inspection_text=text,inspection_image=image)
    return {
        'inspection_merge':result
    }
 
def thermal_merge(state:DDRState) -> DDRState:
    text = state['thermal_text_dignosis']
    image = state['thermal_image_diagnosis']
    result = thermal_merger(thermal_text=text,thermal_image=image)
    return {
        'thermal_merge':result
    }

def  resoning_agent(state:DDRState) -> DDRState:
    impact = state['inspection_merge']
    thermal = state['thermal_merge']
    result = ddr_agent(inspection_merged=impact, thermal_merged=thermal)
    return {
        'ddr_report':result
    }

def save_report(state:DDRState) -> DDRState:
    output = state['ddr_report']
    save_json(output)
    return {}



In [ ]:
graph = StateGraph(DDRState)

graph.add_node("impact_extraction", impact_extraction)
graph.add_node("impact_text", impact_text)
graph.add_node("impact_image", impact_image)
graph.add_node("impact_merge", impact_merge)
 
graph.add_node("thermal_extraction", thermal_extraction)
graph.add_node("thermal_text", thermal_text)
graph.add_node("thermal_image", thermal_image)
graph.add_node("thermal_merge", thermal_merge)

 
graph.add_node("reasoning_agent", resoning_agent)
graph.add_node("save",save_report)

In [ ]:
graph.add_edge(START,'impact_extraction')
graph.add_edge(START,'thermal_extraction')

graph.add_edge("impact_extraction","impact_text")
graph.add_edge("impact_extraction","impact_image")
graph.add_edge("impact_text","impact_merge")
graph.add_edge("impact_image","impact_merge")


graph.add_edge("thermal_extraction","thermal_text")
graph.add_edge("thermal_extraction","thermal_image")
graph.add_edge("thermal_text","thermal_merge")
graph.add_edge("thermal_image","thermal_merge")

graph.add_edge("impact_merge","reasoning_agent")
graph.add_edge("thermal_merge","reasoning_agent")

graph.add_edge("reasoning_agent","save")
graph.add_edge("save",END)

workflow = graph.compile()


In [ ]:
workflow

In [ ]:
res = workflow.invoke({
    "inspection_path" : "Sample Report.pdf",
    "thermal_path"  : "Thermal Images.pdf"
})

In [ ]:
res

In [29]:
from jinja2 import Template
import json

with open("json/ddr.json") as f:
    data = json.load(f)

with open("Report/template.html") as f:
    template = Template(f.read())

html = template.render(
    summary=data.get("property_issue_summary"),
    areas=data.get("areas"),
    additional_notes=data.get("additional_notes"),
    missing_information=data.get("missing_information"),
    conflicts=data.get("conflicts")
)

with open("report.html", "w") as f:
    f.write(html)

In [31]:
import webbrowser
webbrowser.open("report.html")

True

In [ ]:
from src.workflow_graph import workflow 
res = workflow.invoke({
    "inspection_path" : "Sample Report.pdf",
    "thermal_path"  : "Thermal Images.pdf"
})

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by ProtocolError('Connection aborted.', TimeoutError('The write operation timed out')))"))
Content-Length: 4578218
API Key: lsv2_********************************************e2trace=019db41a-2c08-77c2-bd05-3778d8c32661,id=019db41a-f2f2-7102-afda-66368b0468c1; trace=019db41a-2c08-77c2-bd05-3778d8c32661,id=019db41a-f2f3-7513-b222-8354ef951779; trace=019db41a-2c08-77c2-bd05-3778d8c32661,id=019db41a-f2f4-7590-9e95-6820471ff3b3; trace=019db41a-2c08-77c2-bd05-3778d8c32661,id=019db41a-f2f5-7551-9e82-73cee8071cab; trace=019db41a-2c08-77c2-bd05-3778d8c32661,id=019db41a-f2fb-7243-be1d-f62013931064; trace=019db41a-2c08-77c2-bd05-3778d8c32661,id=019db41a-f302-7943-a